# 05 – Task (b): which teams will change coach?

**Business goal:** before the season starts, know which teams will change coach (management, contract planning, media).
**DM goal:** binary classification, one row per team-season, imbalanced (≈ 1 change in 3).

### Which definition of "coach change"?
Our notes from the meetings are not consistent, so the three definitions are built (`src/features.py`) and compared at the end (§11):

| Target | Meaning | Positive rate 2000–09 |
|---|---|---|
| `coach_change_during` | The team has more than one coach during season *t* (mid-season firing) | ≈ 19% |
| `coach_change_end` | The coach who ends season *t* does not start season *t+1* | ≈ 19% |
| **`coach_change_any`** (main) | Either of the two: the coach who opens season *t* is gone by the start of *t+1* | ≈ 31% |

The main analysis uses **`coach_change_any`**, the most complete answer to "will this team change coach this season?". **To confirm with the professor** – changing `TARGET` below re-runs everything for another definition.

**Known the day before season t:** the coach who opens the season (and whether he was hired in the summer, `new_coach`), his history, and the team's results up to t-1.

| Metric | Why |
|---|---|
| **PR-AUC** (main) | With an imbalanced positive class, it focuses on how well we rank the real changes; a random model scores the positive rate (≈ 0.31), not 0.5 |
| ROC-AUC | Probability that a random "change" team gets a higher score than a random "no change" team |
| F1, precision, recall of the positive class | Quality of the final yes/no list; the threshold is chosen on the inner validation to maximise F1 |
| (not) accuracy | "Nobody changes" already gets ≈ 69% accuracy with zero useful information |

In [ ]:
import sys, json, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50, "display.width", 200, "display.precision", 3)
from validation import (TEST_YEARS, walk_forward, classification_metrics, friedman_nemenyi, plot_cd,
                        wilcoxon_vs, FilterWrapperSelector, best_f1_threshold, inner_temporal_cv)
from models import classifiers, _prep
from experiment import run_classification, summary, _fit
from sklearn.base import clone
from imblearn.pipeline import Pipeline
import os
os.makedirs("../data/processed/results", exist_ok=True)

TARGET = "coach_change_any"
df = pd.read_csv("../data/processed/features.csv")
fl = json.load(open("../data/processed/feature_lists.json"))
FEATS = fl["features"]
hist = df[df.is_future == 0].reset_index(drop=True)
future = df[df.is_future == 1].reset_index(drop=True)
lab = hist.dropna(subset=[TARGET])
print("rows with label:", len(lab), "| positive rate 1979-2009:", round(lab[TARGET].mean(), 3),
      "| test seasons:", round(lab[lab.year >= 2000][TARGET].mean(), 3))

In [ ]:
rates = hist.groupby("year")[["coach_change_during", "coach_change_end", "coach_change_any"]].mean()
ax = rates.rolling(3, min_periods=1).mean().plot(figsize=(10, 3.5))
ax.set(ylabel="share of teams (3-season mean)", title="About one team in three changes coach each season; the rate is stable since 1979")
plt.show()

## 1. Baselines
- **Majority class:** "nobody changes" → probability = training positive rate for everyone. PR-AUC = positive rate, F1 = 0.
- **Rule "missed the playoffs":** teams that missed the playoffs at t-1 change coach.
- **Rule "bad team":** score = 1 − points% at t-1 (the worse the team, the higher the risk); threshold: points% < 0.45.

In [ ]:
rows = []
for ty, tr, te in walk_forward(lab):
    y = te[TARGET]
    rules = {"Baseline: majority": (np.full(len(te), tr[TARGET].mean()), 0.99),
             "Baseline: missed playoffs": (1 - te.made_playoffs_lag1.fillna(1).values, 0.5),
             "Baseline: bad team": (1 - te.pts_pct_lag1.values, 0.55)}
    for name, (score, thr) in rules.items():
        rows.append({"model": name, "year": ty, **classification_metrics(y, score, thr)})
base = pd.DataFrame(rows)
summary(base, ["pr_auc", "roc_auc", "f1", "precision", "recall"])

## 2. Imbalanced data: no resampling vs. class weights vs. SMOTE

SMOTE creates synthetic minority examples by interpolating between neighbouring "change" teams. It is a step of the **imblearn `Pipeline`**, so it is applied **only to the training folds** (also inside the inner tuning folds); the test season is never resampled. Applying SMOTE before splitting would leak synthetic copies of test teams into training.

In [ ]:
imb = []
for bal in ["none", "weight", "smote"]:
    zoo = classifiers(bal)
    s, _, _ = run_classification(lab, FEATS, {m: zoo[m] for m in ["LogReg", "RandomForest"]}, TARGET)
    imb.append(s.assign(balance=bal))
imb = pd.concat(imb)
imb.groupby(["model", "balance"])[["pr_auc", "roc_auc", "f1", "precision", "recall"]].mean().round(3)

Ranking metrics (PR-AUC, ROC-AUC) barely change: resampling does not add information, it only moves the scores. What changes is the **threshold behaviour**: without balancing, the models predict few changes (low recall). Because our threshold is tuned for F1 on the inner validation, all three end up close. We use **class weights** where the algorithm supports them (cheaper, no synthetic rows) and **SMOTE** for kNN, Naive Bayes and MLP, which have no `class_weight`.

## 3. Sampling for development and feature selection

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_fixed = lambda: Pipeline(_prep() + [("m", LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))])
lc = []
for w in [3, 5, 10, 15, 20, None]:
    s, _, _ = run_classification(lab, FEATS, {"LR": (lr_fixed(), {})}, TARGET, tune=False, train_window=w)
    lc.append({"train seasons": w or "all", "pr_auc": s.pr_auc.mean(), "train_pr_auc": s.train_pr_auc.mean()})
lc = pd.DataFrame(lc)
ax = lc.plot(x="train seasons", y=["train_pr_auc", "pr_auc"], marker="o", figsize=(7, 3.5))
ax.axhline(base[base.model == "Baseline: majority"].pr_auc.mean(), ls="--", color="grey")
ax.set_title("Learning curve: with few seasons the model overfits; test PR-AUC grows with history"); plt.show(); lc

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, mutual_info_classif
curve = []
for k in [2, 4, 6, 8, 12, 16, 24, len(FEATS)]:
    for kind in ["filter (MI)", "filter + wrapper (MI -> RFE)"]:
        sel = SelectKBest(mutual_info_classif, k=k) if kind.startswith("filter (") else FilterWrapperSelector(k=k, task="classification")
        pipe = Pipeline(_prep() + [("fs", sel), ("m", LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))])
        s, _, _ = run_classification(lab, FEATS, {kind: (pipe, {})}, TARGET, tune=False)
        curve.append({"k": k, "method": kind, "pr_auc": s.pr_auc.mean()})
curve = pd.DataFrame(curve)
ax = sns.lineplot(data=curve, x="k", y="pr_auc", hue="method", marker="o")
ax.axhline(base[base.model == "Baseline: majority"].pr_auc.mean(), ls="--", color="grey", label="random / majority")
ax.set_title("PR-AUC vs number of features (logistic regression)"); ax.legend(); plt.show()
curve.pivot(index="k", columns="method", values="pr_auc")

## 4. Model comparison (8 algorithms)

| Algorithm | Bias | Main parameters | Imbalance |
|---|---|---|---|
| Logistic regression | linear in the log-odds | `C` (less = stronger L2 penalty) | class weights |
| kNN | similar past team-seasons | `k`, weights | SMOTE |
| SVM (RBF) | maximum margin, kernel | `C`, `γ` | class weights |
| Naive Bayes (categorical, on quantile bins) | probabilistic, features independent given the class | `alpha` (smoothing) | SMOTE |
| Decision tree | axis-parallel rules | depth, leaf size | class weights |
| Random Forest | bagging of trees | depth, leaf size, `max_features` | class weights |
| Gradient Boosting | boosting of shallow trees | learning rate, depth | class weights |
| MLP | neural network | hidden layers, `alpha` | SMOTE |

Tuning: grid search on the inner temporal split, scoring = PR-AUC; the selector's k is tuned too.

In [ ]:
import time
zw, zs = classifiers("weight"), classifiers("smote")
zoo = {m: (zs if m in ["kNN", "NaiveBayes", "MLP"] else zw)[m] for m in zw}
t0 = time.time()
scores, preds, params = run_classification(lab, FEATS, zoo, TARGET)
print(f"{time.time()-t0:.0f}s")
allscores = pd.concat([scores, base])
allscores.to_csv("../data/processed/results/task_b_scores.csv", index=False)
preds.to_csv("../data/processed/results/task_b_predictions.csv", index=False)
summary(allscores, ["pr_auc", "roc_auc", "f1", "precision", "recall"])

In [ ]:
order = allscores.groupby("model").pr_auc.mean().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=allscores, x="model", y="pr_auc", order=order, ax=ax, color="#9ecae1")
sns.stripplot(data=allscores, x="model", y="pr_auc", order=order, ax=ax, color="#1f4e79", size=4)
ax.axhline(lab[lab.year >= 2000][TARGET].mean(), ls="--", color="grey"); ax.text(len(order) - 1.5, lab[lab.year >= 2000][TARGET].mean() + .01, "random", color="grey")
ax.tick_params(axis="x", rotation=30); ax.set_title("PR-AUC per test season (dashed = random classifier)")
plt.show()
params.drop(columns="year").groupby("model").agg(lambda s: s.dropna().astype(str).mode().iloc[0] if s.notna().any() else "-")

## 5. Statistical significance

In [ ]:
piv = allscores.pivot(index="year", columns="model", values="pr_auc")
stat, p, ranks, cd = friedman_nemenyi(piv)
print(f"Friedman chi2 = {stat:.2f}, p = {p:.4f}")
fig, ax = plt.subplots(figsize=(9, 4.5)); plot_cd(ranks, cd, ax); plt.show()
models_only = [c for c in piv if not c.startswith("Baseline")]
best = piv[models_only].mean().idxmax()
for b in ["Baseline: majority", "Baseline: bad team", "Baseline: missed playoffs"]:
    w = wilcoxon_vs(piv, best, b)
    print(f"{best} (PR-AUC {piv[best].mean():.3f}) vs {b} ({piv[b].mean():.3f}): Wilcoxon p = {w.pvalue:.3f}, better in {int((piv[best] > piv[b]).sum())}/{len(piv)} seasons")

## 6. Overfitting

In [ ]:
ov = scores.groupby("model")[["train_pr_auc", "pr_auc", "train_roc_auc", "roc_auc"]].mean()
ov["gap_pr_auc"] = ov.train_pr_auc - ov.pr_auc
ov.sort_values("gap_pr_auc", ascending=False)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
def vcurve(make_model, values, scale):
    out = []
    for v in values:
        pipe = Pipeline(_prep(scale=scale) + [("m", make_model(v))])
        s, _, _ = run_classification(lab, FEATS, {"x": (pipe, {})}, TARGET, tune=False)
        out.append({"param": str(v), "train": s.train_pr_auc.mean(), "test": s.pr_auc.mean()})
    return pd.DataFrame(out)
vt = vcurve(lambda d: DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=0), [1, 2, 3, 4, 6, 10, None], False)
vr = vcurve(lambda l: RandomForestClassifier(n_estimators=200, min_samples_leaf=l, class_weight="balanced", random_state=0, n_jobs=-1), [1, 5, 10, 20, 50, 100], False)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
vt.plot(x="param", y=["train", "test"], marker="o", ax=axes[0]); axes[0].set(xlabel="max_depth", ylabel="PR-AUC", title="Tree: train PR-AUC -> 1 with depth, test does not improve")
vr.plot(x="param", y=["train", "test"], marker="o", ax=axes[1]); axes[1].set(xlabel="min_samples_leaf", title="Random Forest: bigger leaves = less memorisation")
plt.show()

## 7. Decision threshold and the final yes/no list

The models output a probability; the client needs a list. The threshold is chosen on the **inner validation seasons** to maximise F1 (never on the test season). If the client prefers catching every change (recall) over avoiding false alarms (precision), the threshold goes down.

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
bp = preds[preds.model == best]
prec, rec, thr = precision_recall_curve(bp[TARGET], bp.proba)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rec, prec, color="#1f4e79"); axes[0].axhline(bp[TARGET].mean(), ls="--", color="grey")
axes[0].set(xlabel="recall", ylabel="precision", title=f"{best}: precision-recall curve (all test seasons)")
ConfusionMatrixDisplay(confusion_matrix(bp[TARGET], bp.pred), display_labels=["stays", "changes"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Confusion matrix with the tuned thresholds"); plt.show()
print("thresholds chosen per season:", scores[scores.model == best].threshold.round(2).tolist())

## 8. Model improvement log

In [ ]:
lr_t = {"LogReg": classifiers("weight", select=False)["LogReg"]}
def mean_pr(feats, models=lr_t):
    s, _, _ = run_classification(lab, feats, models, TARGET)
    return s.pr_auc.mean()
log = [
    ("Baseline: majority (random ranking)", piv["Baseline: majority"].mean()),
    ("Baseline: bad team rule", piv["Baseline: bad team"].mean()),
    ("LogReg on points% t-1 only", mean_pr(["pts_pct_lag1"])),
    ("+ coach tenure and interaction tenure x badness", mean_pr(["pts_pct_lag1", "coach_tenure", "tenure_x_badness"])),
    ("+ all coach features", mean_pr(["pts_pct_lag1"] + [f for f in FEATS if f.startswith(("coach", "new_coach", "franchise", "tenure", "changed"))])),
    ("+ all 44 features", mean_pr(FEATS)),
    ("+ filter/wrapper selection (k tuned)", mean_pr(FEATS, {"LogReg": classifiers("weight")["LogReg"]})),
    (f"best algorithm ({best})", piv[best].mean()),
]
log = pd.DataFrame(log, columns=["change", "mean PR-AUC"]); log["delta"] = log["mean PR-AUC"].diff()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(range(len(log)), log["mean PR-AUC"], marker="o", color="#1f4e79"); ax.set_xticks(range(len(log)), [f"#{i}" for i in range(len(log))])
ax.set_title("Iterations for task (b)"); ax.set_ylabel("mean test PR-AUC"); plt.show(); log

## 9. Interpretation

### 9.1 Logistic regression: odds ratios
Fitted on all labelled seasons, standardised features, 10 features selected. Odds ratio > 1: +1 standard deviation of the feature **increases** the odds of a coach change.

In [ ]:
pre = Pipeline(_prep()).fit(lab[FEATS])
Xs = pre.transform(lab[FEATS]); y = lab[TARGET].astype(int)
sel = FilterWrapperSelector(k=10, task="classification").fit(Xs, y)
lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit(Xs[:, sel.support_], y)
orat = pd.Series(np.exp(lr.coef_[0]), index=np.array(FEATS)[sel.support_]).sort_values()
ax = (orat - 1).plot.barh(figsize=(7, 4), left=1, color=np.where(orat > 1, "#c0392b", "#1f4e79"))
ax.axvline(1, color="k", lw=1); ax.set(xlabel="odds ratio per +1 std", title="What raises (red) or lowers (blue) the risk of a coach change")
plt.show(); orat.to_frame("odds ratio")

### 9.2 A shallow decision tree: rules in plain language

In [ ]:
from sklearn.tree import export_text, plot_tree
from sklearn.impute import SimpleImputer
tf = ["pts_pct_lag1", "coach_tenure", "made_playoffs_lag1", "new_coach", "coach_career_win_pct", "franchise_coaches_5y", "pts_trend", "coach_age"]
Xt = SimpleImputer(strategy="median").fit_transform(lab[tf])
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=40, class_weight="balanced", random_state=0).fit(Xt, y)
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(tree, feature_names=tf, class_names=["stays", "changes"], filled=True, rounded=True, precision=2, fontsize=9, ax=ax); plt.show()
print(export_text(tree, feature_names=tf, decimals=2))
leaves = pd.DataFrame({"leaf": tree.apply(Xt), "y": y.values}).groupby("leaf").y.agg(["size", "mean"]).rename(columns={"size": "teams", "mean": "real change rate"})
leaves.sort_values("real change rate", ascending=False)

### 9.3 Permutation importance of the best model (on the test seasons)

In [ ]:
from sklearn.inspection import permutation_importance
pipe_b, grid_b = zoo[best]
imps = []
for ty, tr, te in walk_forward(lab):
    est = _fit(pipe_b, grid_b, tr[FEATS], tr[TARGET].astype(int), tr.year, "average_precision")
    r = permutation_importance(est, te[FEATS], te[TARGET].astype(int), scoring="average_precision", n_repeats=10, random_state=0)
    imps.append(pd.Series(r.importances_mean, index=FEATS))
imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
ax = imp.head(12).iloc[::-1].plot.barh(figsize=(8, 4.5), color="#1f4e79")
ax.set(xlabel="drop in PR-AUC when the feature is shuffled", title=f"{best}: features it relies on"); plt.show()

## 10. Comparing the three definitions of "coach change"
Same pipeline (logistic regression with selection, and the best model), only the target changes.

In [ ]:
cmp = []
for tgt in ["coach_change_during", "coach_change_end", "coach_change_any"]:
    d = hist.dropna(subset=[tgt])
    for name, z in [("LogReg", {"LogReg": classifiers("weight")["LogReg"]}), (best, {best: zoo[best]})]:
        s, _, _ = run_classification(d, FEATS, z, tgt)
        cmp.append({"target": tgt, "model": name, "pos_rate": s.pos_rate.mean(), "pr_auc": s.pr_auc.mean(),
                    "lift_over_random": s.pr_auc.mean() / s.pos_rate.mean(), "roc_auc": s.roc_auc.mean(), "f1": s.f1.mean()})
pd.DataFrame(cmp).round(3)

## 11. Final predictions: season 2010
The best configuration is re-tuned and retrained on all labelled seasons and gives each team its probability of changing coach in 2010. Assumption: the coach who ended 2009 opens 2010.

In [ ]:
final = _fit(*zoo[best], lab[FEATS], lab[TARGET].astype(int), lab.year, "average_precision")
from experiment import _inner_threshold
thr = _inner_threshold(final, lab, FEATS, TARGET)
future["p_change"] = final.predict_proba(future[FEATS])[:, 1]
future["pred_change"] = np.where(future.p_change >= thr, "yes", "no")
out = future.sort_values("p_change", ascending=False)[["name", "coach", "coach_tenure", "pts_pct_lag1", "made_playoffs_lag1", "p_change", "pred_change"]]
out.to_csv("../data/processed/results/task_b_predictions_2010.csv", index=False)
print(f"threshold = {thr:.2f}; predicted changes: {(future.pred_change == 'yes').sum()} of {len(future)} (historical rate ≈ {lab[TARGET].mean():.0%})")
out

## Conclusions and reflection
_(filled in after looking at the results above – see the guide in `docs/` for the interpretation)_